In [1]:
import torch
import torch.nn as nn
from src.ufno import Net3d
from src.utility import OperatorDataset, load_hdf5
from train_ufno import Config
import numpy as np
import h5py
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
config = Config()

device = config.device
model = Net3d(config.mode1, config.mode2, config.mode3, config.width).to(device)
ckpt = torch.load('checkpoint/ufno.pth')
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print('ufno loaded')

ufno loaded


C:\Users\pc\AppData\Local\Temp\ipykernel_19988\1371158948.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load('checkpoint/ufno.pth')


In [3]:
def load_hdf5_test(file_path):
    print(f">>>>> Loading data from {file_path}")
    data = {}
    with h5py.File(file_path, 'r') as f:
        for key in f.keys():
            val = f[key][-50:]
            data.update({key: val})
            print(f">>>>> Loaded {key} with shape {val.shape}")
    return data

test_dataset = OperatorDataset(load_hdf5_test('dataset/Multi_Cartesian.hdf5'))
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=50, shuffle=False, num_workers=0)

>>>>> Loading data from dataset/Multi_Cartesian.hdf5
>>>>> Loaded permeability with shape (50, 64, 64)
>>>>> Loaded permeability_log with shape (50, 64, 64)
>>>>> Loaded pressure with shape (50, 10, 64, 64)
>>>>> Loaded saturation with shape (50, 10, 64, 64)
>>>>> Loaded time_step with shape (50, 10)
>>>>> Loading min-max values from perm_min_max.json
>>>>> Loading min-max values from ts_min_max.json
>>>>> Loading min-max values from sat_min_max.json
>>>>> Loading min-max values from pre_min_max.json
>>>>> Loaded dataset, x shape: torch.Size([50, 64, 64, 10, 2]), y shape: torch.Size([50, 64, 64, 10, 2])


In [4]:
x_list = []
pred_list = []
y_list = []
def test_model(model, test_loader):
    for i, (x, y) in enumerate(test_loader):
        x = x.to(device)
        y = y.to(device)
        with torch.no_grad():
            pred = model(x)
        pred = pred.cpu().detach().numpy()
        y = y.cpu().detach().numpy()
        x = x.cpu().detach().numpy()
        x_list.append(x)
        pred_list.append(pred)
        y_list.append(y)

test_model(model, test_loader)
pred_list = np.array(pred_list)
y_list = np.array(y_list)
x_list = np.array(x_list)
print(f"x_list shape: {x_list.shape}, pred_list shape: {pred_list.shape}, y_list shape: {y_list.shape}")


x_list shape: (1, 50, 64, 64, 10, 2), pred_list shape: (1, 50, 64, 64, 10, 2), y_list shape: (1, 50, 64, 64, 10, 2)


In [5]:
pred_array = pred_list.reshape(-1, 64, 64, 10, 2)
y_array = y_list.reshape(-1, 64, 64, 10, 2)
x_array = x_list.reshape(-1, 64, 64, 10, 2)

In [ ]:
# def plot_results(x, y, pred, t, idx, ps_string):
#     if ps_string == 'sat':
#         ps = 0
#     elif ps_string == 'pre':
#         ps = 1
#     else:
#         raise ValueError("ps_string must be 'sat' or 'pre'")
#     fig, axs = plt.subplots(1, 3, figsize=(15, 5))
#     axs[0].imshow(x[idx, :, :, t, 0], cmap='jet')
#     axs[0].set_title(f'K @ No.{t} time step')
#     axs[1].imshow(y[idx, :, :, t, ps], cmap='jet')
#     axs[1].set_title(f'Ground Truth of {ps_string}')
#     axs[2].imshow(pred[idx, :, :, t, ps], cmap='jet')
#     axs[2].set_title(f'Prediction of {ps_string}')
#     plt.show()

: 

In [ ]:
plt.imshow(pred_array[0, :, :, 9, 0], cmap='jet')